# 07 - Export and TFLite validation

LiteRT Torch conversion requires **Linux** (Python 3.11 recommended). Keep the
Windows DirectML training environment separate.

1. On Windows, run through **Step 1** to create `artifacts/mobilenetv3_export_cpu.pt`.
2. Open this notebook in a Linux Jupyter kernel or Google Colab. Copy the CPU checkpoint,
   `class_map.json`, `preprocess_contract.json`, and `fine_tune_report.json` into an
   `artifacts` directory beside the notebook. Training images are not required here.
   In Colab, use `/content/artifacts`, or set `AGRILENS_ARTIFACTS_DIR` to your folder.
3. Run all cells in that Linux kernel. The dependency cell installs the converter there.
4. Copy `agrilens_model.tflite`, `model_card.json`, and `tflite_vs_torch_compare.json`
   back to your project artifacts folder. Check that the comparison status is `ok`.

[Official LiteRT Torch setup and conversion API](https://github.com/google-ai-edge/litert-torch/blob/main/README.md)


In [ ]:
import platform
import subprocess
import sys

if platform.system() == "Linux":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "litert-torch", "ai-edge-litert", "torchvision", "pandas", "pillow",
    ])
    print("If packages were already imported, restart the kernel before continuing.")
else:
    print("Windows/macOS: run Step 1 to prepare the CPU checkpoint, then use Linux or Colab.")
    print("LiteRT Torch conversion dependencies will not be installed in this kernel.")


In [ ]:
import json
import importlib
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import mobilenet_v3_small

# Works from the repository root, a sibling directory, or Google Colab.
if os.environ.get("AGRILENS_ARTIFACTS_DIR"):
    ARTIFACTS = Path(os.environ["AGRILENS_ARTIFACTS_DIR"]).expanduser().resolve()
else:
    candidates = [Path.cwd() / "artifacts", Path.cwd() / "agrilens-training" / "artifacts"]
    ARTIFACTS = next((p for p in candidates if p.is_dir()), candidates[0])
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("artifacts directory:", ARTIFACTS)


### Step 1 - load frozen training contract and checkpoint

In [ ]:
class_map_path = ARTIFACTS / "class_map.json"
preprocess_path = ARTIFACTS / "preprocess_contract.json"
checkpoint_path = ARTIFACTS / "mobilenetv3_finetune_best.pt"
report_path = ARTIFACTS / "fine_tune_report.json"

cpu_checkpoint_path = ARTIFACTS / "mobilenetv3_export_cpu.pt"
if not checkpoint_path.exists() and not cpu_checkpoint_path.exists():
    raise FileNotFoundError("Copy mobilenetv3_export_cpu.pt from Windows Step 1 into the artifacts folder.")

for p in [class_map_path, preprocess_path, report_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required artifact: {p}")

class_map = json.loads(class_map_path.read_text(encoding="utf-8"))
preprocess = json.loads(preprocess_path.read_text(encoding="utf-8"))
report = json.loads(report_path.read_text(encoding="utf-8"))
labels = [c["label"] for c in sorted(class_map, key=lambda x: x["id"])]
num_classes = len(labels)

input_size = int(preprocess["input_size"])
mean = preprocess["mean"]
std = preprocess["std"]

print("num_classes:", num_classes)
print("labels:", labels)
print("report best run:", report["best_run"])


In [ ]:
class ExportWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model

    def forward(self, x):
        feats = self.base.features(x)
        pooled = self.base.avgpool(feats)
        emb = torch.flatten(pooled, 1)
        logits = self.base.classifier(emb)
        return logits, emb

model = mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
# Prefer a current, portable checkpoint in the Linux conversion environment.
use_cpu_checkpoint = cpu_checkpoint_path.exists() and (
    not checkpoint_path.exists()
    or cpu_checkpoint_path.stat().st_mtime >= checkpoint_path.stat().st_mtime
)
if use_cpu_checkpoint:
    state_dict = torch.load(cpu_checkpoint_path, map_location="cpu", weights_only=True)
else:
    # Trusted local training artifact only: DirectML checkpoints contain NumPy
    # reconstruction objects unsupported by weights_only in the training PyTorch.
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    state_dict = state["state_dict"] if isinstance(state, dict) and "state_dict" in state else state
    state_dict = {key: value.detach().cpu().contiguous().clone() for key, value in state_dict.items()}
    torch.save(state_dict, cpu_checkpoint_path)
    print("Portable CPU checkpoint saved:", cpu_checkpoint_path)
model.load_state_dict(state_dict)
model.eval()
wrapper = ExportWrapper(model)
print("wrapper ready")


### Step 2 - convert on Linux

Use `litert_torch.convert(wrapper, (sample_input,))`, then export the returned model.
`ai_edge_litert` supplies the inference interpreter; it is not the PyTorch converter.
Conversion errors stop execution instead of silently continuing with a missing export.


In [ ]:
import platform

export_path = ARTIFACTS / "agrilens_model.tflite"
if platform.system() != "Linux":
    raise RuntimeError(
        "LiteRT Torch conversion requires Linux. Step 1 prepared mobilenetv3_export_cpu.pt. "
        "Open this notebook in Linux Jupyter or Google Colab, copy the four artifacts "
        "listed at the top into its artifacts folder, and run all cells there."
    )
try:
    import litert_torch
except ImportError as exc:
    raise RuntimeError(
        "Run the dependency cell in this Linux kernel, restart the kernel if needed, "
        "then rerun this notebook. LiteRT Torch recommends Python 3.11."
    ) from exc

sample_input = torch.randn(1, 3, input_size, input_size)
with torch.no_grad():
    sample_logits, sample_emb = wrapper(sample_input)
    print("sample logits:", sample_logits.shape)
    print("sample embedding:", sample_emb.shape)
    edge_model = litert_torch.convert(wrapper.eval(), (sample_input,))
    edge_model.export(str(export_path))
if not export_path.is_file() or export_path.stat().st_size == 0:
    raise RuntimeError(f"Converter did not create a nonempty TFLite file: {export_path}")
print("exported:", export_path, "bytes:", export_path.stat().st_size)


### Step 3 - if conversion already happened, compare PyTorch and TFLite outputs

This uses a small sample image and checks the logits and embedding agreement between the PyTorch reference and the exported TFLite model.

In [ ]:
def make_reference_image():
    base = np.zeros((input_size, input_size, 3), dtype=np.uint8)
    base[..., 0] = 200
    base[..., 1] = 180
    base[..., 2] = 170
    return torch.tensor(base).permute(2, 0, 1).float() / 255.0

reference_x = make_reference_image().unsqueeze(0)
reference_x = (reference_x - torch.tensor(mean).view(1, -1, 1, 1)) / torch.tensor(std).view(1, -1, 1, 1)
with torch.no_grad():
    ref_logits, ref_emb = wrapper(reference_x)

compare = {
    "reference_logits_shape": list(ref_logits.shape),
    "reference_embedding_shape": list(ref_emb.shape),
    "status": "not_run",
}

if export_path.exists():
    try:
        from ai_edge_litert.interpreter import Interpreter
        interpreter = Interpreter(model_path=str(export_path))
        interpreter.allocate_tensors()
        input_details = interpreter.get_input_details()
        output_details = interpreter.get_output_details()
        tin = reference_x.numpy()
        interpreter.set_tensor(input_details[0]["index"], tin)
        interpreter.invoke()
        # Output ordering can change during conversion. Match by the expected shapes.
        def read_output(expected_shape):
            matches = [d for d in output_details if tuple(d["shape"]) == tuple(expected_shape)]
            if len(matches) != 1:
                raise ValueError(f"Expected one output with shape {tuple(expected_shape)}; got {output_details}")
            return interpreter.get_tensor(matches[0]["index"])
        tflite_logits = read_output(ref_logits.shape)
        tflite_embedding = read_output(ref_emb.shape)

        tflite_logits = torch.tensor(tflite_logits)
        tflite_embedding = torch.tensor(tflite_embedding)

        logits_diff = (ref_logits - tflite_logits).abs().max().item()
        emb_diff = (ref_emb - tflite_embedding).abs().max().item()

        compare = {
            "reference_logits_shape": list(ref_logits.shape),
            "reference_embedding_shape": list(ref_emb.shape),
            "tflite_logits_shape": list(tflite_logits.shape),
            "tflite_embedding_shape": list(tflite_embedding.shape),
            "max_logits_abs_diff": float(logits_diff),
            "max_embedding_abs_diff": float(emb_diff),
            "status": "ok" if (
                np.allclose(ref_logits.numpy(), tflite_logits.numpy(), rtol=1e-3, atol=1e-4)
                and np.allclose(ref_emb.numpy(), tflite_embedding.numpy(), rtol=1e-3, atol=1e-4)
            ) else "mismatch",
        }
        print(json.dumps(compare, indent=2))
    except Exception as exc:
        compare["status"] = f"compare_failed: {type(exc).__name__}: {exc}"
        print(json.dumps(compare, indent=2))

(ARTIFACTS / "tflite_vs_torch_compare.json").write_text(json.dumps(compare, indent=2), encoding="utf-8")
print("saved", ARTIFACTS / "tflite_vs_torch_compare.json")


### Step 4 - model card for app and export metadata


In [ ]:
model_card = {
    "title": "AgriLens plant disease baseline",
    "model_family": "MobileNetV3-Small",
    "class_order": labels,
    "num_classes": num_classes,
    "checkpoint": str(checkpoint_path),
    "input_size": input_size,
    "mean": mean,
    "std": std,
    "resize_policy": preprocess.get("resize_policy", "unspecified"),
    "best_validation_accuracy": report["best_run"]["val_acc"],
    "best_test_accuracy": report["best_run"]["test_acc"],
    "app_interface": {
        "output_logits": f"{num_classes}-class logits",
        "output_embedding": "576-dim embedding before classifier",
        "decision_states": ["possible_match", "uncertain", "unsupported_or_unclear"],
    },
    "notes": [
        "The TFLite export should expose both logits and embedding outputs so rejection can run on-device.",
        "This is a PlantVillage-trained baseline; field performance depends on the real-world photo set and rejection policy."
    ]
}

(ARTIFACTS / "model_card.json").write_text(json.dumps(model_card, indent=2), encoding="utf-8")
print("saved", ARTIFACTS / "model_card.json")
print(json.dumps(model_card, indent=2))
